In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# =================================================================================================
# HYBRID CNN–ViT TRAINING PIPELINE v10
#
# Added improvements:
# 1. Validation evaluation every epoch
# 2. Detailed line-by-line comments
# 3. Mixup accuracy fix preserved
# 4. Same architecture and training strategy
# =================================================================================================

# Install required libraries (Kaggle notebook cell)
!pip install -q timm scikit-learn

# =========================
# IMPORTS
# =========================

import os                     # operating system utilities
import gc                     # garbage collector to free memory if needed
import copy                   # deep copy utility
import random                 # random number generation
import warnings               # warning suppression
import numpy as np            # numerical operations

from PIL import Image         # image loader used by torchvision

import torch                  # main PyTorch library
import torch.nn as nn         # neural network layers
import torch.optim as optim   # optimizers
import torch.nn.functional as F  # functional utilities

from torch.utils.data import DataLoader, Subset     # dataloading utilities
import torchvision.transforms as transforms         # image preprocessing
from torchvision.datasets import ImageFolder        # dataset loader
from torchvision.models import resnet18, ResNet18_Weights  # pretrained ResNet

import timm                  # library containing Vision Transformer models

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

warnings.filterwarnings("ignore")  # suppress non-critical warnings

# =========================
# DEVICE SETUP
# =========================

# Automatically use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

# Enable faster GPU matrix multiplication on newer NVIDIA GPUs
if device.type == "cuda":
    torch.set_float32_matmul_precision("high")

# =========================
# RANDOM SEED (reproducibility)
# =========================

SEED = 42

random.seed(SEED)           # seed Python random
np.random.seed(SEED)        # seed numpy random
torch.manual_seed(SEED)     # seed PyTorch CPU RNG
torch.cuda.manual_seed_all(SEED)  # seed CUDA RNG

# deterministic behavior (slightly slower but reproducible)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# =========================
# HYPERPARAMETERS
# =========================

# emotion class labels
FER_CLASSES = ["angry","disgust","fear","happy","neutral","sad","surprise"]

num_classes = len(FER_CLASSES)

image_size = 224          # input resolution required by ViT
batch_size = 64           # number of images per batch
num_epochs = 70           # maximum training epochs

# differential learning rates
head_lr = 1e-3            # classifier head learns fastest
cnn_lr  = 1e-4            # CNN backbone moderate
vit_lr  = 5e-5            # ViT backbone slowest

dropout_rate = 0.5        # dropout probability

weight_decay  = 3e-4      # L2 regularization
max_grad_norm = 0.5       # gradient clipping threshold

val_frac = 0.2            # 20% validation split

early_stopping_patience = 15

mixup_alpha = 0.2         # mixup strength

tta_n = 5                 # number of test-time augmentations

# AMP disabled due to instability
use_amp = False
amp_device = device.type

# gradient scaler (inactive when AMP disabled)
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

# =========================
# DATA PATHS
# =========================

train_root = "/kaggle/input/datasets/msambare/fer2013/train"
test_root  = "/kaggle/input/datasets/msambare/fer2013/test"

# =========================
# IMAGE TRANSFORMS
# =========================

train_transform = transforms.Compose([

    transforms.Resize((image_size,image_size)),        # resize images

    transforms.Grayscale(num_output_channels=3),       # convert grayscale → 3 channels

    transforms.RandomHorizontalFlip(p=0.5),            # random horizontal flip

    transforms.RandomRotation(15),                     # random rotation

    transforms.RandomAffine(
        degrees=10,
        translate=(0.08,0.08),
        scale=(0.9,1.1)
    ),

    transforms.ColorJitter(brightness=0.15,contrast=0.15),

    transforms.ToTensor(),

    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std=[0.229,0.224,0.225]),

    transforms.RandomErasing(p=0.25,scale=(0.02,0.1))
])

eval_transform = transforms.Compose([

    transforms.Resize((image_size,image_size)),

    transforms.Grayscale(num_output_channels=3),

    transforms.ToTensor(),

    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std=[0.229,0.224,0.225])
])

# =========================
# DATASETS
# =========================

full_train_aug  = ImageFolder(train_root,transform=train_transform)
full_train_eval = ImageFolder(train_root,transform=eval_transform)

full_test       = ImageFolder(test_root,transform=eval_transform)

targets = np.array(full_train_aug.targets)

# =========================
# STRATIFIED SPLIT
# =========================

sss = StratifiedShuffleSplit(n_splits=1,test_size=val_frac,random_state=SEED)

train_idx,val_idx = next(sss.split(np.zeros(len(targets)),targets))

train_dataset = Subset(full_train_aug,train_idx)
val_dataset   = Subset(full_train_eval,val_idx)

test_dataset  = full_test

print("Train:",len(train_dataset))
print("Val:",len(val_dataset))
print("Test:",len(test_dataset))

# =========================
# DATALOADERS
# =========================

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

# =========================
# MIXUP FUNCTIONS
# =========================

def mixup_data(x,y,alpha=0.2):

    lam = np.random.beta(alpha,alpha)

    batch_size = x.size(0)

    index = torch.randperm(batch_size,device=x.device)

    mixed_x = lam*x + (1-lam)*x[index]

    y_a,y_b = y,y[index]

    return mixed_x,y_a,y_b,lam


def mixup_criterion(criterion,pred,y_a,y_b,lam):

    return lam*criterion(pred,y_a)+(1-lam)*criterion(pred,y_b)

# =========================
# MODEL
# =========================

class FERHybridCNNViT(nn.Module):

    def __init__(self,num_classes=7,dropout=0.5):

        super().__init__()

        # CNN backbone (ResNet18)
        self.backbone = resnet18(weights=ResNet18_Weights.DEFAULT)

        self.backbone.fc = nn.Identity()

        # Vision Transformer backbone
        self.vit = timm.create_model(
            "vit_base_patch16_224",
            pretrained=True,
            num_classes=0,
            global_pool="token",
            drop_path_rate=0.1
        )

        # freeze all parameters initially
        for p in self.backbone.parameters():
            p.requires_grad=False

        for p in self.vit.parameters():
            p.requires_grad=False

        # unfreeze deeper CNN layers
        for name,p in self.backbone.named_parameters():

            if "layer3" in name or "layer4" in name:

                p.requires_grad=True

        # unfreeze last transformer blocks
        for name,p in self.vit.named_parameters():

            if any(f"blocks.{i}" in name for i in [9,10,11]) or "norm" in name:

                p.requires_grad=True

        # projection layers
        self.cnn_proj = nn.Sequential(
            nn.Linear(512,512),
            nn.GELU()
        )

        self.vit_proj = nn.Sequential(
            nn.Linear(768,512),
            nn.GELU()
        )

        # fusion head
        self.feature_fusion = nn.Sequential(

            nn.Linear(1024,512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(512,256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(dropout/2)
        )

        self.classifier = nn.Linear(256,num_classes)

    def freeze_bn(self):

        for name,m in self.backbone.named_modules():

            if isinstance(m,nn.BatchNorm2d):

                if not any(tag in name for tag in ["layer3","layer4"]):

                    m.eval()

    def forward(self,x):

        local_feat = self.backbone(x)

        global_feat = self.vit(x)

        local_feat = self.cnn_proj(local_feat)

        global_feat = self.vit_proj(global_feat)

        fused = torch.cat([local_feat,global_feat],dim=1)

        fused = self.feature_fusion(fused)

        logits = self.classifier(fused)

        return logits

model = FERHybridCNNViT(num_classes=num_classes,dropout=dropout_rate).to(device)

# compile model for speed (PyTorch 2.x)
model = torch.compile(model)

# =========================
# LOSS FUNCTION
# =========================

train_labels = [full_train_aug.targets[i] for i in train_idx]

class_counts = np.bincount(train_labels,minlength=num_classes).astype(float)

class_weights = 1.0/np.sqrt(class_counts)

class_weights = class_weights/class_weights.mean()

class_weights = torch.tensor(class_weights,dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights,label_smoothing=0.10)

# =========================
# OPTIMIZER
# =========================

head_params = list(model.feature_fusion.parameters()) + list(model.classifier.parameters())

cnn_params = [p for n,p in model.backbone.named_parameters() if p.requires_grad]

vit_params = [p for n,p in model.vit.named_parameters() if p.requires_grad]

optimizer = optim.AdamW([
    {"params":head_params,"lr":head_lr},
    {"params":cnn_params,"lr":cnn_lr},
    {"params":vit_params,"lr":vit_lr}
],weight_decay=weight_decay)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=num_epochs,
    eta_min=1e-6
)

# =========================
# VALIDATION FUNCTION
# =========================

@torch.no_grad()
def evaluate(loader):

    model.eval()

    total_loss=0
    total_samples=0

    y_true=[]
    y_pred=[]

    for x,y in loader:

        x=x.to(device)
        y=y.to(device)

        logits=model(x)

        loss=criterion(logits,y)

        preds=logits.argmax(1)

        total_loss+=loss.item()*x.size(0)
        total_samples+=y.size(0)

        y_true.extend(y.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

    avg_loss=total_loss/total_samples
    acc=accuracy_score(y_true,y_pred)*100
    macro_f1=f1_score(y_true,y_pred,average="macro")*100

    return avg_loss,acc,macro_f1

# =========================
# TRAINING LOOP
# =========================

best_f1=0
epochs_no_improve=0

for epoch in range(num_epochs):

    model.train()
    model.freeze_bn()

    total_loss=0
    total_correct=0
    total_samples=0

    for x,y in train_loader:

        x=x.to(device)
        y=y.to(device)

        x_mix,y_a,y_b,lam=mixup_data(x,y,mixup_alpha)

        optimizer.zero_grad()

        logits=model(x_mix)

        loss=mixup_criterion(criterion,logits,y_a,y_b,lam)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(),max_grad_norm)

        optimizer.step()

        preds=logits.argmax(1)

        total_correct += lam*(preds==y_a).sum().item() + (1-lam)*(preds==y_b).sum().item()

        total_loss += loss.item()*x.size(0)
        total_samples += y.size(0)

    train_loss = total_loss/total_samples
    train_acc  = 100*total_correct/total_samples

    val_loss,val_acc,val_f1 = evaluate(val_loader)

    scheduler.step()

    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"Train Loss {train_loss:.4f} | Train Acc {train_acc:.2f}% | "
        f"Val Loss {val_loss:.4f} | Val Acc {val_acc:.2f}% | Val F1 {val_f1:.2f}%"
    )

Device: cuda
Train: 22967
Val: 5742
Test: 7178


W0314 13:28:06.704000 2653 torch/_inductor/utils.py:1558] [0/2] Not enough SMs to use max_autotune_gemm mode


Epoch 1/70 | Train Loss 1.6488 | Train Acc 47.95% | Val Loss 1.4179 | Val Acc 60.05% | Val F1 48.33%
Epoch 2/70 | Train Loss 1.5139 | Train Acc 55.49% | Val Loss 1.3369 | Val Acc 64.19% | Val F1 58.40%
Epoch 3/70 | Train Loss 1.4504 | Train Acc 58.38% | Val Loss 1.2974 | Val Acc 65.97% | Val F1 61.06%
Epoch 4/70 | Train Loss 1.4055 | Train Acc 60.72% | Val Loss 1.2823 | Val Acc 66.25% | Val F1 62.03%
Epoch 5/70 | Train Loss 1.3942 | Train Acc 61.32% | Val Loss 1.3162 | Val Acc 65.24% | Val F1 61.36%
Epoch 6/70 | Train Loss 1.3672 | Train Acc 62.75% | Val Loss 1.2349 | Val Acc 68.36% | Val F1 64.63%
Epoch 7/70 | Train Loss 1.3304 | Train Acc 64.44% | Val Loss 1.2595 | Val Acc 68.36% | Val F1 63.84%
